# Phân loại Email Spam
## 9. Đánh giá & So sánh Quy trình Tiền xử lý dữ liệu

### Mục tiêu nghiên cứu
Mục đích của notebook này là đánh giá trực tiếp và đối chiếu khách quan hiệu quả của hai phương pháp xử lý dữ liệu khác nhau trên cùng mô hình **Multinomial Naive Bayes**:
1. **Quy trình tiền xử lý của giải pháp `Navies_vqd` (Tự xây dựng)**: Giữ lại tối đa thông tin, chuẩn hóa có chọn lọc các ký tự đặc biệt nhạy cảm (`!`, `$`) và số thành token đặc biệt.
2. **Quy trình tiền xử lý của thư mục `models` bên ngoài (Nhóm xây dựng chung)**: Thực hiện xóa bỏ hoàn toàn tất cả các dòng khuyết thiếu cột Message (dropna), loại bỏ trùng lặp dựa trên duy nhất cột Message, và loại bỏ toàn bộ dấu câu (Punctuation).

Thông qua việc đối chiếu các chỉ số phân loại (**Accuracy, Precision, Recall, F1-Score, TPR, FPR**), chúng ta sẽ xác minh xem phương pháp xử lý dữ liệu nào tối ưu hơn cho thuật toán Naive Bayes, đặc biệt là khả năng bắt giữ Spam (TPR) và giảm thiểu báo động giả (FPR).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

### 9.1 Định nghĩa lớp Naive Bayes từ đầu (để notebook hoạt động độc lập)

In [ ]:
class NaiveBayesClassifierFromScratch:
    def __init__(self, alpha=1.0, force_alpha=True, fit_prior=True, class_prior=None):
        self.alpha        = alpha
        self.force_alpha  = force_alpha
        self.fit_prior    = fit_prior
        self.class_prior  = class_prior
        self.classes_          = None
        self.class_priors_     = {}
        self.word_likelihoods_ = {}
        self.vocab_size_       = 0

    def get_params(self, deep=True):
        return {
            "alpha": self.alpha,
            "force_alpha": self.force_alpha,
            "fit_prior": self.fit_prior,
            "class_prior": self.class_prior
        }

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes_    = np.unique(y)
        self.vocab_size_ = n_features

        eff_alpha = self.alpha
        if not self.force_alpha and self.alpha < 1e-10:
            eff_alpha = 1e-10

        if self.class_prior is not None:
            for idx, c in enumerate(self.classes_):
                self.class_priors_[c] = self.class_prior[idx]
        elif not self.fit_prior:
            uniform = 1.0 / len(self.classes_)
            for c in self.classes_:
                self.class_priors_[c] = uniform
        else:
            for c in self.classes_:
                self.class_priors_[c] = np.sum(y == c) / n_samples

        X_arr = X.toarray() if hasattr(X, 'toarray') else np.array(X)
        for c in self.classes_:
            X_c    = X_arr[y == c]
            total  = X_c.sum()
            counts = X_c.sum(axis=0)
            self.word_likelihoods_[c] = (
                (counts + eff_alpha) /
                (total  + eff_alpha * self.vocab_size_)
            )
        return self

    def predict(self, X):
        X_arr = X.toarray() if hasattr(X, 'toarray') else np.array(X)
        preds = []
        for row in X_arr:
            scores = {
                c: np.log(self.class_priors_[c]) +
                   np.sum(row * np.log(self.word_likelihoods_[c]))
                for c in self.classes_
            }
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

### 9.2 Tải và huấn luyện trên dữ liệu đã xử lý của `models` bên ngoài

In [ ]:
OUTER_DATA_DIR = '../data/ready_for_train'

X_train_outer = joblib.load(os.path.join(OUTER_DATA_DIR, 'X_train_final.pkl'))
X_test_outer = joblib.load(os.path.join(OUTER_DATA_DIR, 'X_test_final.pkl'))
y_train_outer = joblib.load(os.path.join(OUTER_DATA_DIR, 'y_train.pkl'))
y_test_outer = joblib.load(os.path.join(OUTER_DATA_DIR, 'y_test.pkl'))

y_train_outer = np.array(y_train_outer).flatten()
y_test_outer = np.array(y_test_outer).flatten()

print("Kích thước ma trận đặc trưng của models ngoài:")
print(f"  - X_train: {X_train_outer.shape} (Số lượng mẫu: {X_train_outer.shape[0]:,}, Số đặc trưng: {X_train_outer.shape[1]:,})")
print(f"  - X_test : {X_test_outer.shape} (Số lượng mẫu: {X_test_outer.shape[0]:,}, Số đặc trưng: {X_test_outer.shape[1]:,})")

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Sử dụng GridSearchCV để tìm alpha tối ưu trên dữ liệu của models ngoài
parameters = {'alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]}
grid_outer = GridSearchCV(MultinomialNB(), parameters, cv=3, scoring='f1')
grid_outer.fit(X_train_outer, y_train_outer)

print("Tham số alpha tốt nhất tìm được:", grid_outer.best_params_)

best_clf_outer = grid_outer.best_estimator_
y_pred_outer = best_clf_outer.predict(X_test_outer)

# Tính toán ma trận nhầm lẫn để tìm TPR và FPR (nhãn 1 là Spam, 0 là Ham)
tn_out, fp_out, fn_out, tp_out = confusion_matrix(y_test_outer, y_pred_outer, labels=[0, 1]).ravel()
tpr_out = tp_out / (tp_out + fn_out)
fpr_out = fp_out / (fp_out + tn_out)

metrics_outer = {
    'Accuracy': accuracy_score(y_test_outer, y_pred_outer),
    'Precision': precision_score(y_test_outer, y_pred_outer, pos_label=1),
    'Recall': recall_score(y_test_outer, y_pred_outer, pos_label=1),
    'F1-Score': f1_score(y_test_outer, y_pred_outer, pos_label=1),
    'TPR': tpr_out,
    'FPR': fpr_out
}

### 9.3 Tải và huấn luyện trên dữ liệu đã xử lý của `Navies_vqd`

In [ ]:
OUR_DATA_DIR = './data/ready_for_train'

X_train_our = joblib.load(os.path.join(OUR_DATA_DIR, 'X_train_tfidf.pkl'))
X_test_our = joblib.load(os.path.join(OUR_DATA_DIR, 'X_test_tfidf.pkl'))
y_train_our = joblib.load(os.path.join(OUR_DATA_DIR, 'y_train.pkl'))
y_test_our = joblib.load(os.path.join(OUR_DATA_DIR, 'y_test.pkl'))

y_train_our = np.array(y_train_our).flatten()
y_test_our = np.array(y_test_our).flatten()

print("Kích thước ma trận đặc trưng của Navies_vqd:")
print(f"  - X_train: {X_train_our.shape} (Số lượng mẫu: {X_train_our.shape[0]:,}, Số đặc trưng: {X_train_our.shape[1]:,})")
print(f"  - X_test : {X_test_our.shape} (Số lượng mẫu: {X_test_our.shape[0]:,}, Số đặc trưng: {X_test_our.shape[1]:,})")

In [ ]:
# Huấn luyện mô hình Naive Bayes trên dữ liệu của chúng ta với alpha = 0.001
clf_our = MultinomialNB(alpha=0.001)
clf_our.fit(X_train_our, y_train_our)
y_pred_our = clf_our.predict(X_test_our)

# Tính toán ma trận nhầm lẫn để tìm TPR và FPR (nhãn 'spam' là Spam, 'ham' là Ham)
tn_our, fp_our, fn_our, tp_our = confusion_matrix(y_test_our, y_pred_our, labels=['ham', 'spam']).ravel()
tpr_our = tp_our / (tp_our + fn_our)
fpr_our = fp_our / (fp_our + tn_our)

metrics_our = {
    'Accuracy': accuracy_score(y_test_our, y_pred_our),
    'Precision': precision_score(y_test_our, y_pred_our, pos_label='spam'),
    'Recall': recall_score(y_test_our, y_pred_our, pos_label='spam'),
    'F1-Score': f1_score(y_test_our, y_pred_our, pos_label='spam'),
    'TPR': tpr_our,
    'FPR': fpr_our
}

### 9.4 So sánh đối chiếu trực tiếp hiệu năng

In [ ]:
# Tạo DataFrame so sánh
df_compare = pd.DataFrame({
    'Quy trình Navies_vqd (Tự xây dựng)': pd.Series(metrics_our) * 100,
    'Quy trình models ngoài (Nhóm chung)': pd.Series(metrics_outer) * 100
})
df_compare['Chênh lệch (%)'] = df_compare['Quy trình Navies_vqd (Tự xây dựng)'] - df_compare['Quy trình models ngoài (Nhóm chung)']

print("="*95)
print("             BẢNG SO SÁNH HIỆU NĂNG PHÂN LOẠI EMAIL TRÊN HAI QUY TRÌNH TIỀN XỬ LÝ")
print("="*95)
print(df_compare.round(4).to_string())
print("="*95)

In [ ]:
# Tạo biểu đồ gồm 2 subplots để so sánh trực quan một cách chính xác khoa học
# Do FPR có tỉ lệ rất thấp so với các metric khác (khoảng 1.5% - 2.2%), vẽ chung trên 1 trục y [95, 100] sẽ bị cắt mất.
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2.5, 1]})

# Subplot 1: So sánh Accuracy, Precision, Recall, F1, TPR (Scale cao 95% - 100%)
high_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'TPR']
df_high = df_compare.iloc[:, :2].loc[high_metrics].reset_index().rename(columns={'index': 'Chỉ số'})
df_high_melt = pd.melt(df_high, id_vars=['Chỉ số'], 
                       value_vars=['Quy trình Navies_vqd (Tự xây dựng)', 'Quy trình models ngoài (Nhóm chung)'],
                       var_name='Quy trình', value_name='Phần trăm (%)')

sns.barplot(data=df_high_melt, x='Chỉ số', y='Phần trăm (%)', hue='Quy trình', 
            palette=['#4CAF50', '#FF9800'], edgecolor='black', ax=axes[0])
axes[0].set_ylim(95, 100)
axes[0].set_title('So sánh Chỉ số Hiệu năng Cao (%)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Phần trăm (%)')
axes[0].set_xlabel('Chỉ số đánh giá')

for p in axes[0].patches:
    height = p.get_height()
    if height > 0:
        axes[0].annotate(f'{height:.2f}%', (p.get_x() + p.get_width() / 2., height + 0.08),
                         ha='center', va='bottom', fontsize=9, fontweight='bold')

# Subplot 2: So sánh False Positive Rate (FPR - Scale thấp 0% - 3%)
df_fpr = df_compare.iloc[:, :2].loc[['FPR']].reset_index().rename(columns={'index': 'Chỉ số'})
df_fpr_melt = pd.melt(df_fpr, id_vars=['Chỉ số'], 
                      value_vars=['Quy trình Navies_vqd (Tự xây dựng)', 'Quy trình models ngoài (Nhóm chung)'],
                      var_name='Quy trình', value_name='Phần trăm (%)')

sns.barplot(data=df_fpr_melt, x='Chỉ số', y='Phần trăm (%)', hue='Quy trình', 
            palette=['#4CAF50', '#FF9800'], edgecolor='black', ax=axes[1])
axes[1].set_ylim(0, 3)
axes[1].set_title('So sánh Tỷ lệ Báo động Giả (FPR) (Càng thấp càng tốt)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Phần trăm (%)')
axes[1].set_xlabel('')
axes[1].get_legend().remove() # Ẩn legend ở subplot 2 để tránh trùng lặp

for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f'{height:.2f}%', (p.get_x() + p.get_width() / 2., height + 0.05),
                         ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('So sánh hiệu năng Naive Bayes giữa 2 Quy trình Tiền xử lý dữ liệu (TPR vs FPR)', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('./data', exist_ok=True)
plt.savefig('./data/preprocessing_comparison.png', dpi=300)
plt.show()

### 9.5 Thảo luận chuyên sâu và Nhận định Insight

Dựa trên kết quả đối chiếu hiệu năng từ bảng so sánh và biểu đồ trực quan, ta có thể rút ra những kết luận quan trọng sau:

#### 1. Sự cải thiện đồng thời của cả TPR và FPR
Quy trình tiền xử lý tự xây dựng của **`Navies_vqd`** cho kết quả vượt trội rõ rệt khi cải thiện đồng thời cả 2 chỉ số nhạy cảm:
*   **TPR (True Positive Rate / Recall)**: Tỷ lệ bắt giữ thư rác thành công tăng từ **97.53% lên 98.16%** (tăng **+0.63%**). Nghĩa là lượng thư rác lọt lưới (False Negative) giảm đi đáng kể.
*   **FPR (False Positive Rate)**: Tỷ lệ báo động giả (gán nhầm email thường thành thư rác) giảm mạnh từ **2.25% xuống chỉ còn 1.57%** (giảm **-0.68%**). 
    *   *Ý nghĩa thực tế*: Trong bài toán lọc thư rác, việc gán nhầm email thường thành thư rác là lỗi nghiêm trọng nhất vì làm mất thông tin công việc của người dùng. Việc giảm FPR từ 2.25% xuống 1.57% tương đương mức **giảm tương đối 30.2%** lượng email bị lọc nhầm (báo động giả).

#### 2. Phân tích nguyên nhân kỹ thuật đứng sau sự vượt trội

##### a) Giữ lại và khai thác tốt lượng mẫu dữ liệu (Data Volume)
*   Quy trình ngoài kia sử dụng lệnh xóa bỏ dữ liệu missing một cách quá tay (`dropna(subset=['Message'])`), đồng thời chỉ dựa duy nhất trên cột `Message` để loại trùng lặp (`drop_duplicates(subset=['Message'])`). Việc này khiến tập dữ liệu bị sụt giảm nghiêm trọng từ **33,716 mẫu thô xuống còn 27,383 mẫu** (mất đi **18.6%** dữ liệu huấn luyện).
*   Quy trình `Navies_vqd` dọn dẹp khôn ngoan hơn: chỉ loại bỏ các dòng bị khuyết thiếu đồng thời cả Subject và Message (51 dòng), còn lại giữ nguyên thông tin, điền chuỗi rỗng và ghép hai trường lại. Điều này giúp bảo toàn được **33,665 mẫu** sạch. Tập dữ liệu huấn luyện lớn hơn giúp mô hình Naive Bayes ước lượng xác suất tiên nghiệm ($P(C)$) và xác suất hợp lệ ($P(w|C)$) chính xác và có tính tổng quát hóa cao hơn hẳn.

##### b) Tác dụng của việc giữ lại và mã hóa ký tự đặc biệt (`!` và `$`) thành các token chuyên biệt
*   Quy trình của models bên ngoài thực hiện xóa bỏ toàn bộ dấu câu: `re.sub(f"[{re.escape(string.punctuation)}]", ' ', cleaned)` làm mất sạch các ký tự đặc biệt biểu cảm như dấu chấm than (`!`) và dấu tiền tệ đô-la (`$`).
*   Trong khi đó, ở **Bước 4: EDA** chúng ta đã phát hiện và khai phá insight rằng: dấu chấm than (`!`) xuất hiện trung bình ở Spam nhiều gấp gần **3 lần** so với Ham; và ký tự tiền tệ (`$`) xuất hiện cực kỳ thường xuyên ở email Spam quảng cáo tài chính, dụ dỗ kiếm tiền. 
*   Việc quy trình `Navies_vqd` chuẩn hóa chúng thành các token đặc trưng độc lập `__exclamation__` và `__dollar__` giúp thuật toán Naive Bayes học được xác suất rất cao của các token này gắn liền với nhãn Spam. Điều này đã trực tiếp giúp mô hình tăng **TPR (Recall)** (phát hiện email spam mời chào tài chính nhạy bén hơn) và đồng thời giảm mạnh **FPR** (giảm báo động nhầm các email công việc chứa ký hiệu tiền tệ hoặc chấm than thông thường nhờ bộ từ vựng phong phú).

#### Kết luận
Quy trình tiền xử lý dọn dẹp khôn ngoan và bảo toàn các token đặc trưng biểu cảm (`!`, `$`) của giải pháp `Navies_vqd` đã chứng minh tính hiệu quả vượt trội. Việc thiết kế tiền xử lý trong bài toán phân loại văn bản (đặc biệt là email spam) cần dựa trên phân tích khám phá dữ liệu thực tế (EDA) thay vì chỉ dọn dẹp cơ bản theo lối mòn lý thuyết.